In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

ROOT_DIR = Path("..")

CLEAN_PATH = ROOT_DIR / "data" / "processed" / "complaints_task1_cleaned.csv"
VECTOR_DIR = ROOT_DIR / "vector_store"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

print("Clean file exists:", CLEAN_PATH.exists())
print("Clean file path:", CLEAN_PATH.resolve())
print("Vector store folder:", VECTOR_DIR.resolve())

Clean file exists: True
Clean file path: /Users/mac/rag-complaint-chatbot/data/processed/complaints_task1_cleaned.csv
Vector store folder: /Users/mac/rag-complaint-chatbot/vector_store


/Users/mac/rag-complaint-chatbot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_columns = pd.read_csv(CLEAN_PATH, nrows=0).columns.tolist()
print("Columns in cleaned dataset:")
print(all_columns)

Columns in cleaned dataset:
['date_received', 'original_product', 'issue', 'original_narrative', 'company', 'state', 'company_response', 'timely_response', 'target_product', 'cleaned_narrative', 'cleaned_word_count']


In [3]:
needed_cols = ["target_product", "cleaned_narrative", "cleaned_word_count"]

# Try to use original complaint ID if it exists
id_col = None
for possible_id in ["complaint_id", "Complaint ID", "complaint_id_original"]:
    if possible_id in all_columns:
        id_col = possible_id
        needed_cols.append(possible_id)
        break

clean_df = pd.read_csv(CLEAN_PATH, usecols=needed_cols)

# If original complaint ID is missing, create a fallback ID
if id_col is None:
    clean_df["complaint_id"] = clean_df.index.astype(str)
    print("Warning: Original Complaint ID was not found. Created complaint_id from row index.")
else:
    clean_df = clean_df.rename(columns={id_col: "complaint_id"})

print("Loaded shape:", clean_df.shape)
clean_df.head()

Loaded shape: (563726, 4)


,target_product,cleaned_narrative,cleaned_word_count,complaint_id
0,Credit Card,in good faith i exercised my right and applied...,604,0
1,Credit Card,on last 40 to 42 months they charge me 29 00 p...,43,1
2,Credit Card,i received an email today year that my paypal ...,175,2
3,Money Transfer,never got refund or money to my account was ch...,40,3
4,Savings Account,beware of doing business with american express...,392,4


In [4]:
product_counts = clean_df["target_product"].value_counts()

print(product_counts)
print("Number of product categories:", clean_df["target_product"].nunique())

target_product
Credit Card        229716
Savings Account    180729
Money Transfer     119177
Personal Loan       34104
Name: count, dtype: int64
Number of product categories: 4


In [5]:
SAMPLE_SIZE = 12_000
RANDOM_STATE = 42

counts = clean_df["target_product"].value_counts()
target_sample_counts = (counts / counts.sum() * SAMPLE_SIZE).round().astype(int)

# Fix rounding so total becomes exactly 12,000
difference = SAMPLE_SIZE - target_sample_counts.sum()

while difference != 0:
    if difference > 0:
        product_to_adjust = counts.idxmax()
        target_sample_counts[product_to_adjust] += 1
        difference -= 1
    else:
        product_to_adjust = target_sample_counts.idxmax()
        target_sample_counts[product_to_adjust] -= 1
        difference += 1

print("Target sample counts:")
print(target_sample_counts)
print("Total:", target_sample_counts.sum())

Target sample counts:
target_product
Credit Card        4890
Savings Account    3847
Money Transfer     2537
Personal Loan       726
Name: count, dtype: int64
Total: 12000


In [6]:
sample_parts = []

for product, n_samples in target_sample_counts.items():
    product_df = clean_df[clean_df["target_product"] == product]
    
    sampled_product_df = product_df.sample(
        n=n_samples,
        random_state=RANDOM_STATE
    )
    
    sample_parts.append(sampled_product_df)

sample_df = pd.concat(sample_parts).sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Sample shape:", sample_df.shape)
sample_df["target_product"].value_counts()

Sample shape: (12000, 4)


target_product
Credit Card        4890
Savings Account    3847
Money Transfer     2537
Personal Loan       726
Name: count, dtype: int64

In [7]:
sample_path = ROOT_DIR / "data" / "processed" / "task2_stratified_sample.csv"

sample_df.to_csv(sample_path, index=False)

print("Saved sample to:", sample_path)

Saved sample to: ../data/processed/task2_stratified_sample.csv


In [8]:
original_proportions = clean_df["target_product"].value_counts(normalize=True)
sample_proportions = sample_df["target_product"].value_counts(normalize=True)

comparison = pd.DataFrame({
    "original_proportion": original_proportions,
    "sample_proportion": sample_proportions,
    "sample_count": sample_df["target_product"].value_counts()
})

comparison

,original_proportion,sample_proportion,sample_count
target_product,,,
Credit Card,0.407496,0.407500,4890
Savings Account,0.320597,0.320583,3847
Money Transfer,0.211409,0.211417,2537
Personal Loan,0.060497,0.060500,726


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

experiment_df = sample_df.sample(n=1000, random_state=RANDOM_STATE)

chunk_experiments = [
    {"chunk_size": 300, "chunk_overlap": 50},
    {"chunk_size": 500, "chunk_overlap": 100},
    {"chunk_size": 800, "chunk_overlap": 150},
]

experiment_results = []

for config in chunk_experiments:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len
    )
    
    chunk_counts = []
    
    for text in experiment_df["cleaned_narrative"].astype(str):
        chunks = splitter.split_text(text)
        chunk_counts.append(len(chunks))
    
    experiment_results.append({
        "chunk_size": config["chunk_size"],
        "chunk_overlap": config["chunk_overlap"],
        "avg_chunks_per_complaint": np.mean(chunk_counts),
        "max_chunks_for_one_complaint": np.max(chunk_counts),
        "total_chunks_for_1000_complaints": np.sum(chunk_counts)
    })

chunk_experiment_df = pd.DataFrame(experiment_results)
chunk_experiment_df

,chunk_size,chunk_overlap,avg_chunks_per_complaint,max_chunks_for_one_complaint,total_chunks_for_1000_complaints
0,300,50,4.697,34,4697
1,500,100,3.028,21,3028
2,800,150,1.997,13,1997


In [10]:
FINAL_CHUNK_SIZE = 500
FINAL_CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=FINAL_CHUNK_SIZE,
    chunk_overlap=FINAL_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len
)

chunk_records = []

for row_idx, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    text = str(row["cleaned_narrative"])
    chunks = splitter.split_text(text)
    
    for chunk_idx, chunk_text in enumerate(chunks):
        chunk_records.append({
            "chunk_id": f"{row['complaint_id']}_{chunk_idx}",
            "complaint_id": row["complaint_id"],
            "product": row["target_product"],
            "chunk_index": chunk_idx,
            "chunk_text": chunk_text
        })

chunks_df = pd.DataFrame(chunk_records)

print("Number of original sampled complaints:", len(sample_df))
print("Number of generated chunks:", len(chunks_df))
print("Average chunks per complaint:", len(chunks_df) / len(sample_df))

chunks_df.head()

100%|██████████| 12000/12000 [00:01<00:00, 8057.89it/s]

Number of original sampled complaints: 12000
Number of generated chunks: 35901
Average chunks per complaint: 2.99175


,chunk_id,complaint_id,product,chunk_index,chunk_text
0,344970_0,344970,Credit Card,0,i went to the discover card website today to f...
1,344970_1,344970,Credit Card,1,i went ahead and applied for the credit card a...
2,344970_2,344970,Credit Card,2,applying with anyone else i stated that i alon...
3,344970_3,344970,Credit Card,3,i know how it works and that i have applied fo...
4,344970_4,344970,Credit Card,4,did not appear that the bank had made a mistak...


In [11]:
chunks_path = VECTOR_DIR / "chunk_metadata.csv"

chunks_df.to_csv(chunks_path, index=False)

print("Saved chunk metadata to:", chunks_path)

Saved chunk metadata to: ../vector_store/chunk_metadata.csv


In [12]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = chunks_df["chunk_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 561/561 [01:45<00:00,  5.30it/s]


Embeddings shape: (35901, 384)


In [13]:
embeddings_path = VECTOR_DIR / "chunk_embeddings.npy"

np.save(embeddings_path, embeddings)

print("Saved embeddings to:", embeddings_path)

Saved embeddings to: ../vector_store/chunk_embeddings.npy


In [14]:
import faiss

embeddings = embeddings.astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index size:", index.ntotal)
print("Embedding dimension:", dimension)

FAISS index size: 35901
Embedding dimension: 384


In [15]:
faiss_index_path = VECTOR_DIR / "faiss_index.bin"

faiss.write_index(index, str(faiss_index_path))

print("Saved FAISS index to:", faiss_index_path)

Saved FAISS index to: ../vector_store/faiss_index.bin


In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import faiss

ROOT_DIR = Path("..")
VECTOR_DIR = ROOT_DIR / "vector_store"

chunks_path = VECTOR_DIR / "chunk_metadata.csv"
faiss_index_path = VECTOR_DIR / "faiss_index.bin"

chunks_df = pd.read_csv(chunks_path)
index = faiss.read_index(str(faiss_index_path))

faiss.omp_set_num_threads(1)

print("Chunks loaded:", chunks_df.shape)
print("FAISS vectors:", index.ntotal)
print("Metadata rows:", len(chunks_df))

Chunks loaded: (35901, 5)
FAISS vectors: 35901
Metadata rows: 35901


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

ROOT_DIR = Path("..")
VECTOR_DIR = ROOT_DIR / "vector_store"

chunks_df = pd.read_csv(VECTOR_DIR / "chunk_metadata.csv")
embeddings = np.load(VECTOR_DIR / "chunk_embeddings.npy").astype("float32")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

query = "unauthorized credit card charge and dispute with bank"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores = embeddings @ query_embedding[0]

top_k = 5
top_indices = np.argsort(scores)[-top_k:][::-1]

results = chunks_df.iloc[top_indices].copy()
results["similarity_score"] = scores[top_indices]

results[["similarity_score", "product", "complaint_id", "chunk_text"]]

/Users/mac/rag-complaint-chatbot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6372.16it/s]


,similarity_score,product,complaint_id,chunk_text
7733,0.798834,Credit Card,325157,on the morning of i noticed unauthorized charg...
4892,0.750501,Credit Card,412525,on 19 i went to purchase 400 00 itune gift car...
29620,0.749527,Savings Account,1960,i was charged fraudulently by for 70 00 severa...
303,0.746825,Credit Card,355091,has been harrassing me for payment of a transa...
3585,0.745270,Credit Card,550571,a fraudulent charge was put on my credit card ...


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import faiss

ROOT_DIR = Path("..")
VECTOR_DIR = ROOT_DIR / "vector_store"

sample_path = ROOT_DIR / "data" / "processed" / "task2_stratified_sample.csv"
chunks_path = VECTOR_DIR / "chunk_metadata.csv"
embeddings_path = VECTOR_DIR / "chunk_embeddings.npy"
faiss_index_path = VECTOR_DIR / "faiss_index.bin"

sample_df = pd.read_csv(sample_path)
chunks_df = pd.read_csv(chunks_path)
embeddings = np.load(embeddings_path)
index = faiss.read_index(str(faiss_index_path))

print("Sample complaints:", len(sample_df))
print("Generated chunks:", len(chunks_df))
print("Embedding shape:", embeddings.shape)
print("FAISS index vectors:", index.ntotal)

print("\nSample product distribution:")
print(sample_df["target_product"].value_counts())

print("\nChunk product distribution:")
print(chunks_df["product"].value_counts())

print("\nFiles exist:")
print("Sample file:", sample_path.exists())
print("Chunk metadata:", chunks_path.exists())
print("Embeddings:", embeddings_path.exists())
print("FAISS index:", faiss_index_path.exists())

Sample complaints: 12000
Generated chunks: 35901
Embedding shape: (35901, 384)
FAISS index vectors: 35901

Sample product distribution:
target_product
Credit Card        4890
Savings Account    3847
Money Transfer     2537
Personal Loan       726
Name: count, dtype: int64

Chunk product distribution:
product
Credit Card        15359
Savings Account    11595
Money Transfer      6694
Personal Loan       2253
Name: count, dtype: int64

Files exist:
Sample file: True
Chunk metadata: True
Embeddings: True
FAISS index: True


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import faiss

ROOT_DIR = Path("..")
CLEAN_PATH = ROOT_DIR / "data" / "processed" / "complaints_task1_cleaned.csv"
SAMPLE_PATH = ROOT_DIR / "data" / "processed" / "task2_stratified_sample.csv"
VECTOR_DIR = ROOT_DIR / "vector_store"

CHUNKS_PATH = VECTOR_DIR / "chunk_metadata.csv"
EMBEDDINGS_PATH = VECTOR_DIR / "chunk_embeddings.npy"
FAISS_PATH = VECTOR_DIR / "faiss_index.bin"

print("Files exist:")
print("Cleaned dataset:", CLEAN_PATH.exists())
print("Stratified sample:", SAMPLE_PATH.exists())
print("Chunk metadata:", CHUNKS_PATH.exists())
print("Embeddings:", EMBEDDINGS_PATH.exists())
print("FAISS index:", FAISS_PATH.exists())

sample_df = pd.read_csv(SAMPLE_PATH)
chunks_df = pd.read_csv(CHUNKS_PATH)
embeddings = np.load(EMBEDDINGS_PATH)
index = faiss.read_index(str(FAISS_PATH))

print("\nTask 2 counts:")
print("Sample complaints:", len(sample_df))
print("Generated chunks:", len(chunks_df))
print("Embeddings shape:", embeddings.shape)
print("FAISS index vectors:", index.ntotal)

print("\nSample product distribution:")
print(sample_df["target_product"].value_counts())

print("\nChunk product distribution:")
print(chunks_df["product"].value_counts())

print("\nChecks:")
print("Sample size between 10,000 and 15,000:", 10000 <= len(sample_df) <= 15000)
print("Embeddings match chunks:", embeddings.shape[0] == len(chunks_df))
print("FAISS vectors match chunks:", index.ntotal == len(chunks_df))
print("Embedding dimension is 384:", embeddings.shape[1] == 384)

Files exist:
Cleaned dataset: True
Stratified sample: True
Chunk metadata: True
Embeddings: True
FAISS index: True

Task 2 counts:
Sample complaints: 12000
Generated chunks: 35901
Embeddings shape: (35901, 384)
FAISS index vectors: 35901

Sample product distribution:
target_product
Credit Card        4890
Savings Account    3847
Money Transfer     2537
Personal Loan       726
Name: count, dtype: int64

Chunk product distribution:
product
Credit Card        15359
Savings Account    11595
Money Transfer      6694
Personal Loan       2253
Name: count, dtype: int64

Checks:
Sample size between 10,000 and 15,000: True
Embeddings match chunks: True
FAISS vectors match chunks: True
Embedding dimension is 384: True
